# Zaskaleta AI Twin — 30-Day Germany Short-Film Series
Один постійний AI-клон → 30 мотиваційних відео → Німеччина → 60–90 секунд → 8 сцен на день.
15 із 30 сюжетів містять діалоги: дівчина, друг, колега, співрозмовник або аудиторія/натовп. Ключові фрази завжди говорить головний AI-клон.


In [ ]:
import torch, subprocess, os, json
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Увімкніть T4 GPU')
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
!rm -rf /content/zaskaleta-ai-twin-colab
!git clone --depth 1 https://github.com/sergokharkov/zaskaleta-ai-twin-colab.git /content/zaskaleta-ai-twin-colab
ROOT='/content/zaskaleta-ai-twin-colab'
WORKER=f'{ROOT}/worker'
PLAN=f'{ROOT}/content/monthly_plan_30_days.json'
DIALOGUES=f'{ROOT}/content/dialogue_overrides.json'
MUSETALK='/content/MuseTalk'
VENV_DIR='/content/ai-twin-py311'
PYTHON_BIN=f'{VENV_DIR}/bin/python'
print('✅ Code + 30-day plan + dialogue plan loaded')


In [ ]:
env=os.environ.copy(); env['APP_DIR']=WORKER; env['MUSETALK_ROOT']=MUSETALK; env['VENV_DIR']=VENV_DIR
r=subprocess.run(['bash', f'{WORKER}/install_gpu_engines.sh'], env=env)
if r.returncode != 0: raise RuntimeError(f'Installer failed: {r.returncode}')
print('✅ OpenVoice + MuseTalk ready')


## 1. Google Drive + база клона


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/Zaskaleta_AI_Twin')
BASE.mkdir(parents=True, exist_ok=True)
print('📁', BASE)


In [ ]:
import ipywidgets as widgets
from IPython.display import display
img_ext={'.jpg','.jpeg','.png','.webp'}; aud_ext={'.wav','.mp3','.m4a','.flac'}
photos=sorted([p for p in BASE.iterdir() if p.is_file() and p.suffix.lower() in img_ext])
voices=sorted([p for p in BASE.iterdir() if p.is_file() and p.suffix.lower() in aud_ext])
if len(photos)<5: raise RuntimeError('Додайте мінімум 5 master-фото у MyDrive/Zaskaleta_AI_Twin')
if not voices: raise RuntimeError('Додайте master voice у MyDrive/Zaskaleta_AI_Twin')
photo_pick=widgets.SelectMultiple(options=[(p.name,str(p)) for p in photos], description='5–6 PHOTO:', rows=min(8,len(photos)))
voice_pick=widgets.Dropdown(options=[(p.name,str(p)) for p in voices], description='VOICE:')
display(photo_pick,voice_pick)


In [ ]:
MASTER_PHOTOS=list(photo_pick.value)
if len(MASTER_PHOTOS) not in (5,6): raise ValueError('Виберіть рівно 5 або 6 фото')
VOICE=voice_pick.value
print('✅ MASTER_PHOTOS:', len(MASTER_PHOTOS))
print('✅ VOICE:', VOICE)


## 2. Виберіть день серії


In [ ]:
plan=json.loads(Path(PLAN).read_text(encoding='utf-8'))
dialogue_plan=json.loads(Path(DIALOGUES).read_text(encoding='utf-8'))
dialogue_days={int(k) for k in dialogue_plan.get('days',{}).keys()}
day_pick=widgets.Dropdown(options=[(f"Day {d['day']:02d} — {d['title']} — {d['city']}" + (' 🗣️' if d['day'] in dialogue_days else ''),d['day']) for d in plan['days']], description='DAY:')
display(day_pick)


In [ ]:
DAY=int(day_pick.value)
DAYDIR=BASE/'episodes'/f'Day_{DAY:02d}'
DAYDIR.mkdir(parents=True,exist_ok=True)
subprocess.run([PYTHON_BIN,f'{WORKER}/prepare_daily_episode.py','--plan',PLAN,'--day',str(DAY),'--output-dir',str(DAYDIR),'--dialogues',DIALOGUES],check=True)
episode=json.loads((DAYDIR/'episode.json').read_text(encoding='utf-8'))
print('🎬',episode['title'])
print('📍',episode['city'],'—',episode['location'])
print('👕',episode['outfit'])
print('🗣️ Dialogue:', 'YES' if episode.get('dialogue') else 'NO')
if episode.get('dialogue'):
    print('👥', episode['dialogue'].get('secondary_character'))
    for line in episode['dialogue'].get('dialogues',[]): print(f"  Scene {line['scene']}: {line['speaker']} — {line['text']}")
print('📁',DAYDIR)


## 3. Створити українську озвучку головного героя


In [ ]:
VOICE_OUT=str(DAYDIR/'voiceover_clone.wav')
subprocess.run([PYTHON_BIN,f'{WORKER}/voice_mms_openvoice.py','--script',str(DAYDIR/'voiceover.txt'),'--voice',VOICE,'--output',VOICE_OUT,'--language','uk'],check=True)
print('✅ Main character voiceover:',VOICE_OUT)


## 4. Автоматична озвучка діалогів
Для діалогового дня створюються окремі WAV-файли для кожної репліки. AI_CLONE використовує MASTER_VOICE; другорядні персонажі отримують окремі рольові голосові профілі.


In [ ]:
DIALOGUE_AUDIO_DIR=DAYDIR/'dialogue_audio'
subprocess.run([PYTHON_BIN,f'{WORKER}/generate_dialogue_audio.py','--episode',str(DAYDIR/'episode.json'),'--master-voice',VOICE,'--worker-dir',WORKER,'--python-bin',PYTHON_BIN,'--output-dir',str(DIALOGUE_AUDIO_DIR)],check=True)
if (DIALOGUE_AUDIO_DIR/'dialogue_audio_manifest.json').is_file():
    print('✅ Dialogue audio ready:',DIALOGUE_AUDIO_DIR)
else:
    print('ℹ️ No dialogue audio needed for this day')


## 5. Генерація 8 сцен
В `scene_prompts.json` є повні prompt-и, включно з діалогами. Для справжньої зміни локації/одягу потрібен окремий identity-preserving image/video generation backend. MuseTalk робить lip-sync, але сам не створює нову локацію чи одяг.
Після генерації сцени зберігаються як `scene_01.mp4` … `scene_08.mp4`.


In [ ]:
manifest=json.loads((DAYDIR/'scene_prompts.json').read_text(encoding='utf-8'))
for s in manifest['scenes']:
    print(f"\nSCENE {s['n']:02d} — {s['seconds']} sec — {s['shot']}")
    if s.get('dialogue'):
        for line in s['dialogue']: print(f"  🗣️ {line['speaker']}: {line['text']}")
    print(s['prompt'])
    print('→',DAYDIR/s['expected_clip'])


## 6. Фінальна збірка 60–90 секунд


In [ ]:
FINAL=str(DAYDIR/f'Day_{DAY:02d}_FINAL_9x16.mp4')
subprocess.run([PYTHON_BIN,f'{WORKER}/assemble_daily_episode.py','--episode-dir',str(DAYDIR),'--voice',VOICE_OUT,'--output',FINAL],check=True)
print('✅ FINAL:',FINAL)


In [ ]:
from IPython.display import Video,display
display(Video(FINAL,embed=True,width=360))
